<a href="https://colab.research.google.com/github/kangwonlee/nmisp/blob/main/30_num_int/50_exp_cos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Numerical Integration of a Bessel Function

$$\int_0^\pi{e^{cos{\theta}}}d\theta = \pi I_0(1)$$



* Let's think about the definite integral above.<br>위 정적분을 생각해 보자.
* Can you find its indefinite integral?<br>해당 부정적분을 구할 수 있는가?



* This integral is  closely related to a special type of function called a modified Bessel function of the first kind.<br>이 적분은 수정 제1종 베셀 함수라는 특별한 함수와 밀접히 관련되어 있다고 한다.

$$
\begin{align}
I_n(z)&=\frac{1}{\pi}\int_0^\pi{e^{z cos\theta}cos(n\theta)d\theta} \\
I_0(1)&=\frac{1}{\pi}\int_0^\pi{e^{cos\theta}d\theta}
\end{align}
$$

* Bessel functions are important in physics and engineering due to their ability to describe phenomena with cylindrical or spherical symmetry.<br>베셀함수는 물리학과 공학에서 중요한데, 원통 또는 구면 대칭인 현상을 표현할 수 있기 때문이다.
* It lacks a closed-form solution, requiring exploration of numerical integration techniques.<br>닫힌 해가 없으므로, 수치적인 적분으로 찾아볼 수 밖에 없을 것이다.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import numpy.testing as nt
import scipy.integrate as si



* Integrand<br>적분 대상 함수



In [ ]:
def f(x):
    return np.exp(np.cos(x))



* Let's plot<br>그려보자




In [ ]:
theta_3_rad = np.linspace(-3 * np.pi, 3 * np.pi, (360*3) + 1)
theta_1_rad = np.linspace(0, np.pi, 180 + 1)
y1 = f(theta_1_rad)
y3 = f(theta_3_rad)



In [ ]:
plt.plot(theta_3_rad, y3)
plt.fill_between(theta_1_rad, y1)
plt.xlabel(r'$\theta$')
plt.title(r'$e^{cos{\theta}}$')
plt.grid(True)



* Trapezoid integration<br>사다리꼴 적분



In [ ]:
i_trapz = si.trapezoid(y1, theta_1_rad)
i_trapz



In [ ]:
i_simpson = si.simpson(y1, x=theta_1_rad)
i_simpson



* Two results almost equal?<br>두 결과는 거의 같은가?



In [ ]:
nt.assert_almost_equal(i_trapz, i_simpson)



## 정확한 값과 비교<br>Comparison with the Analytic Value


베셀 함수에는 닫힌 해가 없다고 했는데, 그렇다면 우리 수치 결과가 맞는지 어떻게 알 수 있을까? <br>
We just said the Bessel integral has no closed-form antiderivative &mdash; so how do we know our numerical result is right?

직접 계산은 못 하지만, $I_0$ 자체는 잘 연구된 특수 함수이다. `scipy.special.iv(0, z)` 가 정확한 $I_0(z)$ 값을 매우 높은 정밀도로 제공한다.<br>
We cannot compute it by hand, but $I_0$ itself is a well-studied special function. `scipy.special.iv(0, z)` returns $I_0(z)$ to high precision.


In [ ]:
import scipy.special as sp_special

# 정확한 값 / analytic value
i_exact = np.pi * sp_special.iv(0, 1)
print(f'analytic  pi * I_0(1) = {i_exact:.15f}')
print(f'trapezoid (n=180)     = {i_trapz:.15f}   |err| = {abs(i_trapz - i_exact):.3e}')
print(f'simpson   (n=180)     = {i_simpson:.15f}   |err| = {abs(i_simpson - i_exact):.3e}')


두 수치 적분 결과 모두 *정확한* 값과 잘 일치한다. 이로써 "닫힌 해가 없는 적분에 대해 수치 적분이 *실용적인* 답을 준다" 는 점이 확인된다.<br>
Both numerical estimates agree with the *analytic* value. This is the practical promise: when there is no closed form, numerical integration still gives us a real number we can trust.


## 수렴 차수 확인<br>Convergence Study


사다리꼴 규칙은 $O(h^2)$, 심슨 규칙은 $O(h^4)$ 의 절단 오차를 가진다. 분할 수 $n$ 을 두 배로 했을 때 오차가 어떻게 줄어드는지 직접 확인해 보자.<br>
The trapezoid rule has $O(h^2)$ truncation error; Simpson's rule has $O(h^4)$. Let's see what doubling the number of panels actually does to the error.


In [ ]:
n_list = np.array([4, 8, 16, 32, 64, 128, 256, 512])

err_trapz, err_simpson = [], []
for n in n_list:
    theta = np.linspace(0, np.pi, n + 1)
    y = f(theta)
    err_trapz.append(  abs(si.trapezoid(y, theta)     - i_exact))
    err_simpson.append(abs(si.simpson(y,  x=theta)    - i_exact))

err_trapz   = np.array(err_trapz)
err_simpson = np.array(err_simpson)

# 각 단계에서 오차가 줄어드는 비율 / per-step error reduction ratio
print('  n   |     trapezoid err      ratio   |     simpson err        ratio')
print('------+-------------------------------+---------------------------------')
for i, n in enumerate(n_list):
    rt = '   --  ' if i == 0 else f'{err_trapz[i-1]/err_trapz[i]:7.3f}'
    rs = '   --  ' if i == 0 else f'{err_simpson[i-1]/err_simpson[i]:7.3f}' if err_simpson[i] > 0 else '   inf '
    print(f'{n:4d}  |  {err_trapz[i]:.3e}    {rt}  |  {err_simpson[i]:.3e}     {rs}')

plt.figure(figsize=(7, 5))
plt.loglog(n_list, err_trapz,   'o-', label='trapezoid')
plt.loglog(n_list, err_simpson, 's-', label="Simpson's 1/3")
# 기울기 참조선 / reference slopes
plt.loglog(n_list, err_trapz[0]   * (n_list[0] / n_list)**2, 'k:',  alpha=0.5, label='$O(n^{-2})$')
plt.loglog(n_list, err_simpson[0] * (n_list[0] / n_list)**4, 'k--', alpha=0.5, label='$O(n^{-4})$')
plt.xlabel('n  (number of panels)')
plt.ylabel('|error|')
plt.title(r'$\pi I_0(1) = \int_0^\pi e^{\cos\theta}\,d\theta$  &mdash;  convergence')
plt.grid(True, which='both', alpha=0.3)
plt.legend()
plt.show()


사다리꼴 오차의 비율이 약 4 배씩 줄어드는 것 ($h$ 가 절반이면 오차가 $1/4$, 즉 $O(h^2)$) 과 심슨 오차가 약 16 배씩 줄어드는 것 ($O(h^4)$) 을 직접 볼 수 있다. 같은 자료점 수에서 심슨 규칙이 압도적으로 정확하다.<br>
We can see directly that trapezoid error shrinks by roughly $\times 4$ each step (halving $h$ &rArr; quartering error, i.e. $O(h^2)$), while Simpson shrinks by roughly $\times 16$ ($O(h^4)$). At the same number of samples, Simpson's rule is dramatically more accurate.


## 응용: 원통 좌표계의 물리 문제<br>Application: Physics Problems in Cylindrical Coordinates


베셀 함수가 단순히 수학적 호기심이 아닌 이유:<br>
Why Bessel functions are not just a mathematical curiosity:

* **원통 안의 열전도 / Heat conduction in a cylinder.** 무한히 긴 원통 안의 정상 상태 온도 분포는 라플라스 방정식의 분리 변수 해를 통해 베셀 함수 $J_0(\lambda r), I_0(\lambda r), \ldots$ 로 표현된다.<br>
  The steady-state temperature in an infinite cylinder, via separation of variables on Laplace's equation, is expressed in $J_0, I_0$, etc.
* **원형 막의 진동 / Vibration of a circular drum.** 원형 막의 진동 모드 (예: 팀파니 북) 는 베셀 함수의 영점에서 결정된다.<br>
  The normal modes of a circular membrane (a kettle drum, for instance) are determined by zeros of $J_n$.
* **광섬유의 도파 모드 / Waveguide modes in optical fibers.** 원통형 도파관의 전자기장 분포는 베셀 함수로 기술된다.<br>
  Electromagnetic field profiles in cylindrical waveguides are Bessel-function profiles.

이러한 문제에서 *정확한* 적분식은 닫힌 형태가 없을 수 있고, 수치 적분이 실용적인 수학적 도구가 된다 &mdash; 우리가 방금 본 것과 정확히 같은 방법으로.<br>
In each, the exact integrals may have no closed form, and numerical integration is the practical tool &mdash; using the very methods we just demonstrated.


**베셀 함수의 모습 / What the Bessel functions look like.** &nbsp; 우리가 위에서 적분으로 계산한 $I_0(1)$ 은 이 함수족 $\{I_n(z)\}_{n=0,1,2,\ldots}$ 의 한 점이다. 같은 적분 공식 $I_n(z) = \frac{1}{\pi}\int_0^\pi e^{z\cos\theta}\cos(n\theta)\,d\theta$ 가 모든 $n$ 에 대해 성립한다 &mdash; *닫힌 해가 없는 적분이 함수족 전체를 정의한다.*<br>
The $I_0(1)$ we computed by integration is one point in the family $\{I_n(z)\}_{n=0,1,2,\ldots}$. The same integral formula $I_n(z) = \frac{1}{\pi}\int_0^\pi e^{z\cos\theta}\cos(n\theta)\,d\theta$ holds for every $n$ &mdash; *an integral with no closed form defines an entire function family*.


In [ ]:
x = np.linspace(0, 4, 200)

plt.figure(figsize=(8, 4.5))
for n in range(4):
    plt.plot(x, sp_special.iv(n, x), label=f'$I_{n}(x)$')
plt.axvline(1.0, color='gray', linestyle=':', alpha=0.5, label='$x = 1$ (computed above)')
plt.xlabel('$x$')
plt.ylabel('$I_n(x)$')
plt.title('Modified Bessel functions of the first kind')
plt.grid(True)
plt.legend()
plt.show()


## 연습 문제<br>Exercises


Try this 1: $I_1(1) = \frac{1}{\pi}\int_0^\pi e^{\cos\theta}\cos\theta\,d\theta$ 을 사다리꼴/심슨 규칙으로 계산하고, `scipy.special.iv(1, 1)` 과 비교하시오.<br>
Compute $I_1(1) = \frac{1}{\pi}\int_0^\pi e^{\cos\theta}\cos\theta\,d\theta$ using the trapezoid and Simpson rules, and compare with `scipy.special.iv(1, 1)`.


Try this 2: $z \in [0, 5]$ 에서 $I_0(z) = \frac{1}{\pi}\int_0^\pi e^{z\cos\theta}\,d\theta$ 의 표를 만들고, `scipy.special.iv(0, z)` 와 비교하시오.<br>
Tabulate $I_0(z) = \frac{1}{\pi}\int_0^\pi e^{z\cos\theta}\,d\theta$ for $z \in [0, 5]$ and compare with `scipy.special.iv(0, z)`.


Try this 3: 위 수렴 그래프에서, $z = 5$ 일 때 (적분값이 훨씬 더 큼) $n$ 별 오차가 어떻게 달라지는지 비교하시오. $z$ 가 커지면 같은 정밀도를 위해 더 많은 분할이 필요한가?<br>
Repeat the convergence study above for $z = 5$ (where the integral value is much larger). Does larger $z$ demand more panels for the same precision? Why?


## 참고문헌<br>References


* M. Abramowitz, I. A. Stegun (eds.), *Handbook of Mathematical Functions*, Dover, 1965 (Ch. 9 *Bessel Functions of Integer Order*).
* M. L. Boas, *Mathematical Methods in the Physical Sciences*, 3rd Ed., Wiley, 2006 (Ch. 12 *Series Solutions of Differential Equations; Bessel Functions*).
* R. L. Burden, J. D. Faires, A. M. Burden, *Numerical Analysis*, 10th Ed., Cengage, 2016 (Ch. 4 *Numerical Differentiation and Integration*).
* SciPy documentation, [`scipy.special.iv`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.special.iv.html).


## Final Bell<br>마지막 종


In [ ]:
# stackoverfow.com/a/24634221
import os
os.system("printf '\a'");
